In [1]:
from pathlib import Path
from ai_news_digest.config import Settings
from ai_news_digest.pipeline import collect_articles, rank_and_select, summarize_and_build, _get_llm
from datetime import datetime

In [2]:
# Parámetros de corrida
DAYS = 30
TOP_N = 20 ##15
#ENSURE_SOURCES = [] 
ENSURE_SOURCES = ['Kdnuggets.com']
MONTH_NAME = "marzo_26"
LANG = "es"

ts = datetime.now().strftime("%Y%m%d_%H%M%S")
OUT_PATH = "outputs/Highlights_AI" + "_"+ MONTH_NAME + "_" + ts +".txt"
print(f"Output path: {OUT_PATH}")

settings = Settings()

Output path: outputs/Highlights_AI_marzo_26_20260331_095215.txt


In [3]:
# Cliente LLM compartido — acumula tokens entre ranking y summary
llm_client = _get_llm(settings)
print(f"LLM client: {type(llm_client).__name__ if llm_client else 'None'}")

Trying OpenRouterClient...
LLM client: OpenRouterClient


In [4]:
settings

Settings(newsapi_key='7061081eafba4ec4919593e8560e6307', openai_api_key='', openai_model_relevance='gpt-4o-mini', openai_model_summary='gpt-4o-mini', openrouter_api_key='OPENROUTER_KEY_REDACTED', openrouter_model_relevance='nvidia/nemotron-3-super-120b-a12b:free', openrouter_model_summary='stepfun/step-3.5-flash:free', azure_api_key='', azure_endpoint='', azure_deployment_relevance='', azure_deployment_summary='', azure_api_version='2024-02-01')

## Articles Collect

In [5]:
print("🔎 [1/4] Recolectando artículos...")
df_all = collect_articles(settings, days=DAYS)
print(f"   → Artículos recolectados: {len(df_all)}")
if df_all.empty:
        print("   ⚠️  No se encontraron artículos.")

df_all.head(5)

🔎 [1/4] Recolectando artículos...
Total NewsAPI domains used: 125
From date:  2026-03-01 to date:  2026-03-31
------Artículos en Ingles recolectados con NewsAPI: 54
Total NewsAPI domains used: 125
From date:  2026-03-01 to date:  2026-03-31
------Artículos en Español recolectados con NewsAPI: 1
Procesando https://techcrunch.com/category/artificial-intelligence/...
Procesando https://techcrunch.com/category/artificial-intelligence/page/2/...
Procesando https://techcrunch.com/category/artificial-intelligence/page/3/...
Procesando https://techcrunch.com/category/artificial-intelligence/page/4/...
Procesando https://techcrunch.com/category/artificial-intelligence/page/5/...
------Artículos recolectados con TechCrunch: 152
   → Artículos recolectados: 207


,fuente,titulo,url,fecha,contenido
0,The Verge,Why people really hate AI,https://www.theverge.com/podcast/897900/ai-tru...,2026-03-20T13:27:53Z,is editor-at-large and Vergecast co-host with ...
1,The Verge,Inside the secret meeting that led to the AI p...,https://www.theverge.com/ai-artificial-intelli...,2026-03-04T06:06:52Z,"In early January, a group of 90 or so politica..."
2,The Verge,DLSS 5: Has Nvidia’s AI graphics technology go...,https://www.theverge.com/games/896518/nvidia-d...,2026-03-18T12:25:06Z,Nvidia has revealed a new “3D guided neural re...
3,The Verge,Future Sony PlayStation games will use AI to i...,https://www.theverge.com/news/898283/future-so...,2026-03-20T19:49:59Z,is a senior editor and founding member of The ...
4,The Verge,Amazon is making an Alexa phone,https://www.theverge.com/tech/897915/amazon-tr...,2026-03-20T13:42:51Z,"Over 10 years after shelving the Fire Phone, A..."


In [6]:
df_all.fuente.value_counts()

fuente
TechCrunch    152
The Verge      34
Wired          20
BBC News        1
Name: count, dtype: int64

In [7]:
df_all.to_csv("outputs/all_articles_mar_26_v2.csv", index=False)

## Articles Ranking

In [ ]:
# import pandas as pd
# df_all = pd.read_csv("outputs/all_articles_mar_26.csv")
# df_all.fuente.value_counts()
# df_all.head(5)

,fuente,titulo,url,fecha,contenido
0,The Verge,Why people really hate AI,https://www.theverge.com/podcast/897900/ai-tru...,2026-03-20T13:27:53Z,is editor-at-large and Vergecast co-host with ...
1,The Verge,Inside the secret meeting that led to the AI p...,https://www.theverge.com/ai-artificial-intelli...,2026-03-04T06:06:52Z,"In early January, a group of 90 or so politica..."
2,The Verge,DLSS 5: Has Nvidia’s AI graphics technology go...,https://www.theverge.com/games/896518/nvidia-d...,2026-03-18T12:25:06Z,Nvidia has revealed a new “3D guided neural re...
3,The Verge,Future Sony PlayStation games will use AI to i...,https://www.theverge.com/news/898283/future-so...,2026-03-20T19:49:59Z,is a senior editor and founding member of The ...
4,The Verge,Amazon is making an Alexa phone,https://www.theverge.com/tech/897915/amazon-tr...,2026-03-20T13:42:51Z,"Over 10 years after shelving the Fire Phone, A..."


In [8]:
print("📊 [2/4] Rankeando artículos por relevancia...")
df_top = rank_and_select(df_all, settings, top_n=TOP_N, ensure_source=ENSURE_SOURCES, llm_client=llm_client)
print(f"   → Seleccionados top {len(df_top)} artículos")
if llm_client and hasattr(llm_client, "token_summary"):
    print("   ", llm_client.token_summary())

📊 [2/4] Rankeando artículos por relevancia...
ensure_source: ['Kdnuggets.com']
Using LLM: OpenRouterClient
client: <ai_news_digest.llm.openrouter.OpenRouterClient object at 0x00000256CBB50E90>
   Batch LLM scoring: 207 artículos en 11 batches de 20...
   Batch 1/11...
   Batch 2/11...
   Batch 3/11...
   Batch 4/11...
   Batch 5/11...
   Batch 6/11...
   Batch 7/11...
   Batch 8/11...
   Batch 9/11...
   Batch 10/11...
   Batch 11/11...
   Scoring: 207 con LLM, 0 con heurística
   → Seleccionados top 20 artículos
    Tokens usados — prompt: 16,332 | completion: 29,943 | total: 46,275


C:\Users\clara.o.villalba\Documents\Medium\MediumAutomation-1\ai_news_digest\pipeline.py:158: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df_limited = df_final.groupby('fuente', group_keys=False).apply(lambda x: x.head(10)).reset_index(drop=True)


In [9]:
df_top.fuente.value_counts()

fuente
TechCrunch    10
Wired          6
The Verge      4
Name: count, dtype: int64

In [10]:
df_top.to_csv("outputs/top_ranked_articles_mar_26_v2.csv", index=False)

## Article Sumarize

In [11]:
# import pandas as pd
# df_top = pd.read_csv("outputs/top_ranked_articles_mar_26.csv")
# print(df_top.fuente.value_counts())
# df_top.head(5)

In [12]:
print("📝 [3/4] Generando resúmenes y armando artículo...")
article, df_res = summarize_and_build(df_top, settings, month_name=MONTH_NAME, lang=LANG, llm_client=llm_client)
print("   → Resúmenes generados")
if llm_client and hasattr(llm_client, "token_summary"):
    print("   ", llm_client.token_summary())

📝 [3/4] Generando resúmenes y armando artículo...
Using LLM for summarization: OpenRouterClient
   [1/20] Resumiendo: Mantis Biotech is making ‘digital twins’ of humans to help solve medic...
   [1/20] OK
   [2/20] Resumiendo: ByteDance’s new AI video generation model, Dreamina Seedance 2.0, come...
Error summarizing 'ByteDance’s new AI video generation model, Dreamina Seedance': 429 Client Error: Too Many Requests for url: https://openrouter.ai/api/v1/chat/completions
   [3/20] Resumiendo: Google unveils TurboQuant, a new AI memory compression algorithm — and...
   [3/20] OK
   [4/20] Resumiendo: Nvidia is quietly building a multibillion-dollar behemoth to rival its...
   [4/20] OK
   [5/20] Resumiendo: ByteDance’s AI Ambitions Are Being Hampered by Compute Restraints and ...
   [5/20] OK
   [6/20] Resumiendo: Cohere launches an open source voice model specifically for transcript...
   [6/20] OK
   [7/20] Resumiendo: Nvidia’s DLSS 5 uses generative AI to boost photorealism in video ga

In [13]:
df_res.resumen

0     **Mantis Biotech desarrolla 'gemelos digitales...
1     (Sin LLM) OpenAI may be dialing back its effor...
2     **Google presenta TurboQuant: el algoritmo de ...
3     **Nvidia Construye en Silencio un Gigante de R...
4     **ByteDance lanza Seedance 2.0, pero las restr...
5     **Cohere presenta Transcribe: modelo de voz de...
6     **Nvidia Presenta DLSS 5: IA Generativa para E...
7     **DLSS 5: La IA de Nvidia redibuja el renderiz...
8     **Spotify lanza herramienta para proteger perf...
9     **Google presenta Lyria 3 Pro: generación musi...
10    **Mistral impulsa la competencia en síntesis d...
11    **Título:** Gimlet Labs resuelve el cuello de ...
12    **Claude en el Campo de Batalla: Cómo Palantir...
13    **Palantir apuntala su dominio en IA militar c...
14    (Sin LLM) Google announced on Tuesday that all...
15    **Título: Nvidia apuesta por los agentes autón...
16    **Título: Smack Technologies impulsa la IA mil...
17    **Los modelos de IA a medida marcan un nue

In [14]:
article

'# 🧠 Los Highlights de marzo_26 en Inteligencia Artificial\n\nTe compartimos las noticias más destacadas del mes.\n\n### **Mantis Biotech desarrolla \'gemelos digitales\' humanos para superar la escasez de datos biomédicos**\n\n![**Mantis Biotech desarrolla \'gemelos digitales\' humanos para superar la escasez de datos biomédicos**](https://techcrunch.com/wp-content/uploads/2019/09/brain_pills.jpg?resize=1200,899)\n\n**Mantis Biotech desarrolla \'gemelos digitales\' humanos para superar la escasez de datos biomédicos**\n\nEl potencial de los grandes modelos de lenguaje (LLM) entrenados con vastos conjuntos de datos para revolucionar la investigación biomédica —desde la genómica y el descubrimiento de fármacos hasta el diagnóstico en tiempo real— choca con un cuello de botella fundamental: la escasez de datos fiables y representativos en casos límite, como enfermedades raras o condiciones atípicas. Esta brecha de disponibilidad de datos limita la capacidad de estos sistemas para general

In [15]:
print("💾 [4/4] Guardando resultados...")
Path(OUT_PATH).write_text(article, encoding="utf-8")
csv_path = Path(OUT_PATH).with_suffix(".csv")
csv_path.write_text(df_res.to_csv(index=False), encoding="utf-8")
print(f"   → Guardado {OUT_PATH} y {csv_path.name}")

print("✅ Pipeline finalizado con éxito.")

💾 [4/4] Guardando resultados...
   → Guardado outputs/Highlights_AI_marzo_26_20260331_095215.txt y Highlights_AI_marzo_26_20260331_095215.csv
✅ Pipeline finalizado con éxito.


In [16]:
# Total de tokens usados en todo el pipeline
if llm_client and hasattr(llm_client, "token_summary"):
    print("📊 Token usage total del pipeline:")
    print("  ", llm_client.token_summary())
else:
    print("No LLM client disponible — no hay tokens que reportar.")

📊 Token usage total del pipeline:
   Tokens usados — prompt: 30,416 | completion: 49,409 | total: 79,825
